# Chapter 16 &mdash; 2-SAT in Polynomial Time: the Implication Graph

**Concept 6 of the Chapter 16 decomposition:** *2-SAT in Polynomial Time: the Implication Graph*

A 2-clause is a pair of implications; the formula is unsatisfiable iff some variable and its negation share a strongly connected component.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16/Concept-2SAT-In-Polynomial-Time/Concept-2SAT-In-Polynomial-Time.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


$(a \vee b)$ is exactly $(\neg a \Rightarrow b) \wedge (\neg b \Rightarrow a)$. So a
2-CNF formula **is** a directed graph on the $2n$ literals, with two edges per clause.

> **Theorem.** A 2-CNF is unsatisfiable **iff** some variable $x$ has $x$ and $\neg x$
> in the same strongly connected component of that graph.

Because then $x \Rightarrow \neg x$ and $\neg x \Rightarrow x$, and no truth value
survives.

SCCs are computable in **linear time** (Tarjan, Kosaraju), so 2-SAT is in $P$. And the
SCC order even hands you a satisfying assignment: process components in reverse
topological order, setting each undecided literal true.

Adding a third literal per clause destroys this: $(a\vee b\vee c)$ is not a pair of
implications, and the graph structure is gone.

## 2. Definitions

### The implication graph

In [ ]:
# --- a tiny CNF toolkit -------------------------------------------------
# A literal is an int: 3 means x3, -3 means NOT x3.
# A clause is a tuple of literals; a formula is a list of clauses.
from itertools import product

def nvars(F):
    return max((abs(l) for c in F for l in c), default=0)

def evaluate(F, assign):
    # assign: dict var -> bool
    return all(any(assign[abs(l)] == (l > 0) for l in c) for c in F)

def brute_sat(F):
    n = nvars(F)
    for bits in product([False, True], repeat=n):
        a = {i + 1: bits[i] for i in range(n)}
        if evaluate(F, a): return a
    return None

def show_cnf(F):
    def lit(l): return ("x%d" % l) if l > 0 else ("~x%d" % -l)
    return " AND ".join("(" + " OR ".join(lit(l) for l in c) + ")" for c in F)


def implication_graph(F):
    # each 2-clause (a, b) gives -a -> b and -b -> a
    g = {}
    for c_ in F:
        assert len(c_) == 2, "2-SAT only"
        a, b = c_
        g.setdefault(-a, set()).add(b)
        g.setdefault(-b, set()).add(a)
    for l in list(g):
        g.setdefault(-l, set())
    return g

def sccs(g):
    # Kosaraju
    order, seen = [], set()
    def dfs1(u):
        seen.add(u)
        for v in g.get(u, ()): 
            if v not in seen: dfs1(v)
        order.append(u)
    for u in list(g):
        if u not in seen: dfs1(u)
    rg = {}
    for u in g:
        rg.setdefault(u, set())
        for v in g[u]: rg.setdefault(v, set()).add(u)
    comp, cid = {}, 0
    def dfs2(u, cid):
        comp[u] = cid
        for v in rg.get(u, ()):
            if v not in comp: dfs2(v, cid)
    for u in reversed(order):
        if u not in comp:
            dfs2(u, cid); cid += 1
    return comp

### The decision procedure, and the assignment it yields

In [ ]:
def sat2(F):
    g = implication_graph(F)
    comp = sccs(g)
    n = nvars(F)
    for v in range(1, n + 1):
        if v in comp and -v in comp and comp[v] == comp[-v]:
            return None, v                    # unsatisfiable, witness v
    # reverse topological order = larger component id first in Kosaraju
    a = {}
    for v in range(1, n + 1):
        a[v] = comp.get(v, 0) > comp.get(-v, 0)
    return a, None

## 3. Tests

A satisfiable 2-CNF, and its implication graph.

In [ ]:
F = [(1, 2), (-1, 3), (-2, -3)]
print(show_cnf(F))
g = implication_graph(F)
for u in sorted(g, key=abs):
    if g[u]: print("   %-4s -> %s" % (u, sorted(g[u])))

The SCC test says satisfiable, and hands back an assignment.

In [ ]:
a, bad = sat2(F)
print("assignment :", a, " unsat witness :", bad)
assert a is not None and evaluate(F, a)
print("evaluates to True? ", evaluate(F, a))

An **unsatisfiable** 2-CNF: $x$ and $\neg x$ in one component.

In [ ]:
U = [(1, 1), (-1, -1)]        # x  AND  not x
a, bad = sat2(U)
comp = sccs(implication_graph(U))
print(show_cnf(U))
print("component of  x1 :", comp.get(1))
print("component of ~x1 :", comp.get(-1))
print("satisfiable? ", a is not None, " witness variable :", bad)
assert a is None and bad == 1

The procedure agrees with brute force on random instances.

In [ ]:
import random
def random_2sat(nvar, nclause, seed):
    random.seed(seed)
    F = []
    for _ in range(nclause):
        v1, v2 = random.sample(range(1, nvar + 1), 2)
        F.append((v1 * random.choice([1, -1]), v2 * random.choice([1, -1])))
    return F

bad = 0
for s in range(60):
    F = random_2sat(5, 8, s)
    a, _ = sat2(F)
    brute = brute_sat(F)
    if (a is not None) != (brute is not None): bad += 1
    if a is not None: assert evaluate(F, a), (F, a)
print("60 random 2-SAT instances : %d disagreements with brute force" % bad)
assert bad == 0

**It is polynomial** &mdash; linear, in fact, in clauses plus variables.

In [ ]:
import time
for n in [50, 200, 800]:
    F = random_2sat(n, 3 * n, 1)
    t0 = time.time(); sat2(F); t1 = time.time()
    print("  %4d vars, %4d clauses : %.4fs" % (n, 3 * n, t1 - t0))
print("\nBrute force at n=50 would inspect 2^50 assignments.")

**Why the trick dies at three literals.**

In [ ]:
print("(a OR b)        ==  (~a => b) AND (~b => a)          -- two edges")
print("(a OR b OR c)   ==  (~a => (b OR c)) ...             -- not an edge")
print()
print("An implication whose consequent is a DISJUNCTION is not a graph edge,")
print("so there is no implication graph, no SCC test, and no polynomial")
print("algorithm.  3-SAT is NP-complete (Concept 8).")

## 4. Exercises


1. Draw the implication graph for $(x_1\vee x_2)\wedge(\neg x_1 \vee x_2)\wedge(\neg x_2 \vee x_3)$.
2. Prove the "same SCC" condition is necessary as well as sufficient.
3. Why does the reverse-topological assignment rule work?

In [ ]:
# Your work for the exercises above.